# 📈 Nifty 50 Stock Trading Recommendation System
## Deep Reinforcement Learning (PPO) + Technical Analysis + Sentiment Analysis + Market Regime

**Framework:** Custom PPO (Proximal Policy Optimization) via Stable-Baselines3  
**Market:** Indian Stock Market (NSE — Yahoo Finance .NS suffix)  
**Output:** BUY / HOLD / SELL recommendation per selected stock

### Pipeline:
1. User selects one stock from 10 curated Nifty 50 tickers
2. Download OHLCV data for **all 10 stocks** (2016–present)
3. EDA: Visualise price, volume, returns, correlation across all 10 stocks
4. Compute technical indicators → table + composite strength score
5. Market Regime Detection on ^NSEI index (HMM) → score + Bull/Bear/Sideways label
6. Sentiment Analysis via **Finnhub + NewsAPI** (real data, no synthetic)
7. PPO trains on **all 10 stocks** combined; evaluation on selected stock
8. Backtest: Sharpe, Sortino, Calmar, Win Rate, Max Drawdown, Precision
9. Final output table: Technical Score | Sentiment | Regime | PPO Signal | Top 5 News

---

## CELL 1 — Install All Dependencies

In [ ]:
# ============================================================
# CELL 1: INSTALL DEPENDENCIES
# Run once. Restart runtime after if on Google Colab.
# ============================================================

!pip install -q yfinance pandas numpy matplotlib seaborn
!pip install -q stable-baselines3[extra]
!pip install -q gymnasium
!pip install -q ta
!pip install -q hmmlearn
!pip install -q requests
!pip install -q scikit-learn
!pip install -q vaderSentiment
!pip install -q newsapi-python

print("✅ All packages installed!")


✅ All packages installed!


## CELL 2 — Imports

In [ ]:
# ============================================================
# CELL 2: IMPORTS
# ============================================================

import warnings
warnings.filterwarnings('ignore')

# Core
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # non-GUI backend so inline display works everywhere
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
import seaborn as sns
from datetime import datetime, timedelta
import os
import time
import requests

# Data
import yfinance as yf

# Technical Indicators
import ta
from ta.trend import MACD, SMAIndicator, EMAIndicator, ADXIndicator
from ta.momentum import RSIIndicator, StochasticOscillator, ROCIndicator
from ta.volatility import BollingerBands, AverageTrueRange
from ta.volume import OnBalanceVolumeIndicator, VolumeWeightedAveragePrice

# Sentiment
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
try:
    from newsapi import NewsApiClient
    NEWSAPI_AVAILABLE = True
except ImportError:
    NEWSAPI_AVAILABLE = False
    print("⚠  newsapi-python not found. Install with: pip install newsapi-python")

# Market Regime
from hmmlearn import hmm

# RL
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback
from stable_baselines3.common.monitor import Monitor

# ML utilities
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import precision_score

print("✅ All imports successful!")
print(f"Run date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")


## CELL 3 — Configuration & Stock Selection

In [ ]:
# ============================================================
# CELL 3: CONFIGURATION & USER STOCK SELECTION
# ============================================================

# ---- API KEYS (replace with your actual keys) ----
FINNHUB_API_KEY  = "d7m2le9r01qk7lvusehgd7m2le9r01qk7lvusei0"   # https://finnhub.io (free tier)
NEWSAPI_KEY      = "eb055041f5e14c299c593e6e3ce1ec25"                         # https://newsapi.org (free tier)

# ---- 10 CURATED NIFTY 50 STOCKS ----
STOCK_OPTIONS = {
    "1":  ("RELIANCE.NS",   "Reliance Industries"),
    "2":  ("TCS.NS",        "Tata Consultancy Services"),
    "3":  ("HDFCBANK.NS",   "HDFC Bank"),
    "4":  ("INFY.NS",       "Infosys"),
    "5":  ("ICICIBANK.NS",  "ICICI Bank"),
    "6":  ("HINDUNILVR.NS", "Hindustan Unilever"),
    "7":  ("ITC.NS",        "ITC Limited"),
    "8":  ("SBIN.NS",       "State Bank of India"),
    "9":  ("BHARTIARTL.NS", "Bharti Airtel"),
    "10": ("KOTAKBANK.NS",  "Kotak Mahindra Bank"),
}

# All 10 tickers used for PPO training
ALL_TICKERS = [(v[0], v[1]) for v in STOCK_OPTIONS.values()]

print("\n========================================")
print("   NIFTY 50 STOCK RECOMMENDATION SYSTEM")
print("========================================")
print("\nAvailable stocks:")
for k, (ticker, name) in STOCK_OPTIONS.items():
    print(f"  [{k:>2}] {ticker:<20} {name}")

user_choice = input("\nEnter stock number (1-10): ").strip()
if user_choice not in STOCK_OPTIONS:
    user_choice = "1"
    print("Invalid choice. Defaulting to RELIANCE.NS")

SELECTED_TICKER, SELECTED_NAME = STOCK_OPTIONS[user_choice]
# Finnhub symbol format (remove .NS, uppercase)
FINNHUB_SYMBOL = "BSE:" + SELECTED_TICKER.replace(".NS", "")

print(f"\n✅ Selected: {SELECTED_NAME} ({SELECTED_TICKER})")
print(f"   PPO trains on ALL 10 Nifty stocks for generalization.")
print(f"   Evaluation / recommendation is for: {SELECTED_TICKER} only.")

# ---- DATE RANGES ----
TRAIN_START = "2016-01-01"
TRAIN_END   = "2023-12-31"
VAL_START   = "2024-01-01"
VAL_END     = "2024-12-31"
TEST_START  = "2025-01-01"
TEST_END    = datetime.now().strftime("%Y-%m-%d")  # today

FULL_START  = TRAIN_START
FULL_END    = TEST_END

# ---- TRADING PARAMETERS ----
TRANSACTION_COST = 0.001   # 0.1% brokerage
MAX_SHARES       = 100
DUMMY_CAPITAL    = 1000000  # internal RL environment only

# ---- PPO HYPERPARAMETERS (updated per user spec) ----
TOTAL_TIMESTEPS  = 150000   # ↓ 70% reduction
LEARNING_RATE    = 0.0003   # slightly higher → faster learning
N_STEPS          = 512      # ↓ faster updates
BATCH_SIZE       = 64       # ↓ faster training
N_EPOCHS         = 5        # ↓ major speed gain
GAMMA            = 0.99
GAE_LAMBDA       = 0.95
CLIP_RANGE       = 0.2      # default PPO
ENT_COEF         = 0.01     # slightly more exploration
VF_COEF          = 0.5
MAX_GRAD_NORM    = 0.5

# ---- TECHNICAL INDICATOR WINDOWS ----
RSI_WINDOW  = 14
MACD_FAST   = 12
MACD_SLOW   = 26
MACD_SIGNAL = 9
BB_WINDOW   = 20
SMA_SHORT   = 10
SMA_LONG    = 50
ATR_WINDOW  = 14
ADX_WINDOW  = 14
ROC_WINDOW  = 10

# ---- MARKET REGIME ----
N_REGIMES       = 3   # Bull=2, Sideways=1, Bear=0
REGIME_LOOKBACK = 60

# ---- RANDOM SEED ----
SEED = 42
np.random.seed(SEED)

print(f"\n📅 Train:  {TRAIN_START}  →  {TRAIN_END}")
print(f"   Val:    {VAL_START}  →  {VAL_END}")
print(f"   Test:   {TEST_START}  →  {TEST_END}")
print(f"   Algorithm : PPO (updated hyperparameters)")
print(f"   Total timesteps: {TOTAL_TIMESTEPS:,}")
print("✅ Configuration complete!")


## CELL 4 — Download Stock Data (All 10 Stocks + Nifty Index)

In [ ]:
# ============================================================
# CELL 4: DOWNLOAD STOCK DATA — ALL 10 TICKERS + NIFTY INDEX
# PPO trains on all 10 stocks; evaluation uses selected stock
# ============================================================

def download_stock_data(ticker, start, end):
    """Downloads OHLCV data for one NSE ticker via yfinance."""
    print(f"📥 Downloading {ticker} from {start} to {end}...")
    df = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=True)
    if df.empty:
        print(f"  ⚠  No data for {ticker}, skipping.")
        return pd.DataFrame()
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df = df.reset_index()
    df.columns = [str(c).lower().strip() for c in df.columns]
    if 'date' not in df.columns:
        df.rename(columns={'index': 'date'}, inplace=True)
    df['date'] = pd.to_datetime(df['date'])
    df['ticker'] = ticker
    df = df.sort_values('date').reset_index(drop=True)
    print(f"  ✅ {len(df)} rows")
    return df


# Download all 10 stocks
all_raw_data = {}
for ticker, name in ALL_TICKERS:
    df = download_stock_data(ticker, FULL_START, FULL_END)
    if not df.empty:
        all_raw_data[ticker] = df

# Selected stock raw data (used for display, tech indicators, sentiment)
raw_data = all_raw_data[SELECTED_TICKER]

# ---- Display Data Summary for Selected Stock ----
print(f"\n{'='*60}")
print(f"  DATA SUMMARY: {SELECTED_NAME} ({SELECTED_TICKER})")
print(f"{'='*60}")
print(f"  Total rows   : {len(raw_data):,}")
print(f"  Date range   : {raw_data['date'].min().date()} → {raw_data['date'].max().date()}")
print(f"  Columns      : {list(raw_data.columns)}")
print(f"\n  Latest 5 rows:")
display_cols = ['date','open','high','low','close','volume']
display_df = raw_data[display_cols].tail(5).copy()
for col in ['open','high','low','close']:
    display_df[col] = display_df[col].round(2)
display_df['volume'] = display_df['volume'].apply(lambda x: f"{int(x):,}")
print(display_df.to_string(index=False))

print(f"\n  Statistical Summary:")
print(raw_data[['open','high','low','close','volume']].describe().round(2).to_string())

print(f"\n  All 10 stocks downloaded:")
for tkr, df in all_raw_data.items():
    print(f"    {tkr:<20} {len(df)} rows")

# Download Nifty 50 Index for regime detection
print(f"\n📥 Downloading ^NSEI (Nifty 50 Index) for regime detection...")
nifty_index = download_stock_data("^NSEI", FULL_START, FULL_END)
print("✅ Data download complete!")


## CELL 4B — Exploratory Data Analysis (EDA)

In [ ]:
# ============================================================
# CELL 4B: EXPLORATORY DATA ANALYSIS (EDA)
# Visualise price, volume, returns and cross-stock correlation
# ============================================================

sns.set_theme(style='darkgrid', palette='muted')

# ── 1. Price History — Selected Stock ────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)
fig.suptitle(f'EDA: {SELECTED_NAME} ({SELECTED_TICKER})', fontsize=14, fontweight='bold')

axes[0].plot(raw_data['date'], raw_data['close'], color='#2980b9', linewidth=1)
axes[0].fill_between(raw_data['date'], raw_data['low'], raw_data['high'], alpha=0.15, color='#3498db')
axes[0].set_ylabel('Price (₹)')
axes[0].set_title('Close Price with High-Low Band')

vol_color = ['#27ae60' if raw_data['close'].iloc[i] >= raw_data['open'].iloc[i] else '#e74c3c'
             for i in range(len(raw_data))]
axes[1].bar(raw_data['date'], raw_data['volume'], color=vol_color, alpha=0.7, width=1)
axes[1].set_ylabel('Volume')
axes[1].set_title('Trading Volume (Green=Up Day, Red=Down Day)')

daily_ret = raw_data['close'].pct_change() * 100
axes[2].fill_between(raw_data['date'], daily_ret, 0,
                     where=daily_ret >= 0, color='#27ae60', alpha=0.6, label='Positive')
axes[2].fill_between(raw_data['date'], daily_ret, 0,
                     where=daily_ret < 0,  color='#e74c3c', alpha=0.6, label='Negative')
axes[2].set_ylabel('Daily Return (%)')
axes[2].set_title('Daily Return (%)')
axes[2].legend()

plt.tight_layout()
plt.show()

# ── 2. Return Distribution ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Return Distribution — {SELECTED_NAME}', fontsize=13, fontweight='bold')

rets = daily_ret.dropna()
axes[0].hist(rets, bins=80, color='#3498db', alpha=0.8, edgecolor='white')
axes[0].axvline(rets.mean(), color='#2ecc71', linewidth=2, label=f'Mean: {rets.mean():.3f}%')
axes[0].axvline(rets.mean() - 2*rets.std(), color='#e74c3c', linestyle='--', linewidth=1.5,
                label=f'−2σ: {rets.mean()-2*rets.std():.2f}%')
axes[0].axvline(rets.mean() + 2*rets.std(), color='#e74c3c', linestyle='--', linewidth=1.5,
                label=f'+2σ: {rets.mean()+2*rets.std():.2f}%')
axes[0].set_xlabel('Daily Return (%)');  axes[0].set_ylabel('Frequency')
axes[0].set_title('Daily Return Histogram'); axes[0].legend()

# Rolling 30-day volatility
rolling_vol = rets.rolling(30).std()
axes[1].plot(raw_data['date'].iloc[len(raw_data)-len(rolling_vol):], rolling_vol.values,
             color='#9b59b6', linewidth=1.2)
axes[1].set_ylabel('30-day Rolling Std Dev (%)')
axes[1].set_title('Rolling 30-day Volatility')
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout();  plt.show()

# ── 3. All 10 Stocks Normalised Close (base 100) ─────────────
fig, ax = plt.subplots(figsize=(16, 6))
colors_map = plt.cm.tab10(np.linspace(0, 1, len(all_raw_data)))
for (tkr, df), clr in zip(all_raw_data.items(), colors_map):
    base = df['close'].iloc[0]
    ax.plot(df['date'], df['close'] / base * 100, linewidth=1.2,
            label=tkr.replace('.NS',''), color=clr)
ax.set_title('All 10 Stocks — Normalised Close Price (Base 100 at 2016-01-01)',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Normalised Price (Base=100)')
ax.legend(ncol=5, fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout(); plt.show()

# ── 4. Cross-stock Return Correlation Heatmap ────────────────
ret_dict = {}
for tkr, df in all_raw_data.items():
    s = df.set_index('date')['close'].pct_change()
    ret_dict[tkr.replace('.NS','')] = s
ret_df = pd.DataFrame(ret_dict).dropna()

fig, ax = plt.subplots(figsize=(10, 8))
corr = ret_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            vmin=-1, vmax=1, ax=ax, linewidths=0.5,
            cbar_kws={"shrink": 0.8})
ax.set_title('Cross-Stock Daily Return Correlation (All 10 Nifty Stocks)',
             fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

# ── 5. Monthly Return Heatmap — Selected Stock ───────────────
sel_close = raw_data.set_index('date')['close']
monthly_ret = sel_close.resample('ME').last().pct_change() * 100
monthly_df  = monthly_ret.reset_index()
monthly_df.columns = ['date', 'return']
monthly_df['year']  = monthly_df['date'].dt.year
monthly_df['month'] = monthly_df['date'].dt.strftime('%b')
pivot = monthly_df.pivot_table(index='year', columns='month', values='return')
month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
pivot = pivot.reindex(columns=[m for m in month_order if m in pivot.columns])

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn', center=0,
            ax=ax, linewidths=0.3, cbar_kws={"label": "Monthly Return (%)"})
ax.set_title(f'Monthly Return Heatmap — {SELECTED_NAME}', fontsize=12, fontweight='bold')
ax.set_ylabel('Year'); ax.set_xlabel('')
plt.tight_layout(); plt.show()

print("✅ EDA complete!")


## CELL 5 — Technical Indicators + Indicator Table + Composite Score

In [ ]:
# ============================================================
# CELL 5: TECHNICAL INDICATORS + INDICATOR TABLE + SCORE
# ============================================================

def add_technical_indicators(df):
    """Computes all technical indicators for a single-stock DataFrame."""
    df = df.copy().sort_values('date')
    close  = df['close']
    high   = df['high']
    low    = df['low']
    volume = df['volume']

    # RSI
    rsi_ind = RSIIndicator(close=close, window=RSI_WINDOW)
    df['rsi'] = rsi_ind.rsi()

    # MACD
    macd_ind = MACD(close=close, window_fast=MACD_FAST,
                    window_slow=MACD_SLOW, window_sign=MACD_SIGNAL)
    df['macd']        = macd_ind.macd()
    df['macd_signal'] = macd_ind.macd_signal()
    df['macd_hist']   = macd_ind.macd_diff()

    # Bollinger Bands
    bb = BollingerBands(close=close, window=BB_WINDOW)
    df['bb_upper']  = bb.bollinger_hband()
    df['bb_lower']  = bb.bollinger_lband()
    df['bb_middle'] = bb.bollinger_mavg()
    df['bb_width']  = bb.bollinger_wband()
    df['bb_pct']    = bb.bollinger_pband()

    # SMA
    df['sma_10'] = SMAIndicator(close=close, window=SMA_SHORT).sma_indicator()
    df['sma_50'] = SMAIndicator(close=close, window=SMA_LONG).sma_indicator()

    # EMA
    df['ema_20'] = EMAIndicator(close=close, window=20).ema_indicator()

    # ATR
    df['atr'] = AverageTrueRange(high=high, low=low, close=close,
                                  window=ATR_WINDOW).average_true_range()

    # OBV
    df['obv'] = OnBalanceVolumeIndicator(close=close, volume=volume).on_balance_volume()

    # ADX
    adx_ind = ADXIndicator(high=high, low=low, close=close, window=ADX_WINDOW)
    df['adx'] = adx_ind.adx()

    # Stochastic
    stoch = StochasticOscillator(high=high, low=low, close=close)
    df['stoch_k'] = stoch.stoch()
    df['stoch_d'] = stoch.stoch_signal()

    # ROC
    df['roc'] = ROCIndicator(close=close, window=ROC_WINDOW).roc()

    # Price-based features
    df['daily_return']  = close.pct_change()
    df['log_return']    = np.log(close / close.shift(1))
    df['volatility_10'] = df['daily_return'].rolling(10).std()
    df['close_norm']    = (close - close.rolling(50).mean()) / (close.rolling(50).std() + 1e-8)

    print(f"✅ Technical indicators computed. Shape: {df.shape}")
    return df


def compute_technical_strength_score(df):
    """Computes composite technical strength score (0–100)."""
    df = df.copy()
    signals = pd.DataFrame(index=df.index)

    signals['rsi_sig']       = df['rsi'].clip(0, 100) / 100
    macd_h = df['macd_hist']
    signals['macd_sig']      = ((macd_h - macd_h.min()) / (macd_h.max() - macd_h.min() + 1e-8)).clip(0, 1)
    signals['bb_sig']        = df['bb_pct'].clip(0, 1)
    signals['sma_cross_sig'] = (df['close'] > df['sma_50']).astype(float)
    signals['ema_sig']       = (df['close'] > df['ema_20']).astype(float)
    signals['adx_sig']       = ((df['adx'] - 10) / 40).clip(0, 1)
    signals['stoch_sig']     = df['stoch_k'].clip(0, 100) / 100
    roc = df['roc']
    signals['roc_sig']       = ((roc - roc.min()) / (roc.max() - roc.min() + 1e-8)).clip(0, 1)
    obv_ma = df['obv'].rolling(20).mean()
    signals['obv_sig']       = (df['obv'] > obv_ma).astype(float)
    atr_norm = (df['atr'] / (df['close'] + 1e-8))
    signals['atr_sig']       = (1 - atr_norm.clip(0, 0.1) / 0.1).clip(0, 1)

    df['tech_strength_score'] = signals.mean(axis=1) * 100

    signal_cols = ['rsi_sig','macd_sig','bb_sig','sma_cross_sig',
                   'ema_sig','adx_sig','stoch_sig','roc_sig','obv_sig','atr_sig']
    for col in signal_cols:
        df[col] = signals[col]

    return df, signal_cols


# Run on selected stock
data_with_ti = add_technical_indicators(raw_data)
data_with_ti, SIGNAL_COLS = compute_technical_strength_score(data_with_ti)

# ── Latest values for display
latest = data_with_ti.dropna(subset=['tech_strength_score']).iloc[-1]
score  = latest['tech_strength_score']
signal_label = ("STRONG BULLISH" if score >= 70 else
               ("BULLISH"        if score >= 55 else
               ("NEUTRAL"        if score >= 45 else
               ("BEARISH"        if score >= 30 else "STRONG BEARISH"))))

# ── INDICATOR TABLE ──────────────────────────────────────────
indicator_meta = {
    'RSI (14)'           : ('rsi',       latest.get('rsi', np.nan),       0,   100,  'rsi_sig',       30,  70,   "Oversold <30 | Overbought >70"),
    'MACD Histogram'     : ('macd_hist', latest.get('macd_hist', np.nan), None,None, 'macd_sig',      0.4, 0.6,  "Positive = Bullish Momentum"),
    'BB %B'              : ('bb_pct',    latest.get('bb_pct', np.nan),    0,   1,    'bb_sig',        0.2, 0.8,  "0=Lower Band, 1=Upper Band"),
    'SMA Cross (10/50)'  : ('sma_10',    latest.get('close', np.nan),     None,None, 'sma_cross_sig', 0.5, 0.5,  "Close>SMA50 = Bullish"),
    'EMA Cross (20)'     : ('ema_20',    latest.get('ema_20', np.nan),    None,None, 'ema_sig',       0.5, 0.5,  "Close>EMA20 = Bullish"),
    'ADX (14)'           : ('adx',       latest.get('adx', np.nan),       0,   100,  'adx_sig',       20,  25,   "Trend Strength: >25 Strong"),
    'Stochastic %K'      : ('stoch_k',   latest.get('stoch_k', np.nan),   0,   100,  'stoch_sig',     20,  80,   "Oversold <20 | Overbought >80"),
    'ROC (10)'           : ('roc',       latest.get('roc', np.nan),       None,None, 'roc_sig',       0.4, 0.6,  "Positive = Upward Momentum"),
    'OBV Trend'          : ('obv',       latest.get('obv', np.nan),       None,None, 'obv_sig',       0.5, 0.5,  "OBV>20MA = Bullish Volume"),
    'ATR Relative'       : ('atr',       latest.get('atr', np.nan),       None,None, 'atr_sig',       0.4, 0.6,  "Low Rel ATR = Lower Volatility"),
}

print(f"\n{'='*90}")
print(f"  TECHNICAL INDICATOR TABLE — {SELECTED_NAME}  |  Latest: {latest['date'].date()}")
print(f"{'='*90}")
header = f"{'Indicator':<22} {'Raw Value':>12} {'Normalised':>12} {'Range Status':>14} {'Interpretation'}"
print(header)
print("-"*90)
for ind_name, (col, raw_val, lo, hi, sig_col, low_thresh, high_thresh, interp) in indicator_meta.items():
    norm_val = float(latest.get(sig_col, 0))
    if norm_val >= high_thresh:
        status = "✅ GOOD"
    elif norm_val <= low_thresh:
        status = "⚠ CAUTION"
    else:
        status = "🔶 NEUTRAL"
    raw_str = f"{raw_val:.2f}" if raw_val is not None and not (isinstance(raw_val, float) and np.isnan(raw_val)) else "N/A"
    print(f"{ind_name:<22} {raw_str:>12} {norm_val:>12.3f} {status:>14}  {interp}")
print("-"*90)
print(f"  ▶  FINAL TECHNICAL SCORE : {score:.1f} / 100   |   Signal: {signal_label}")
print(f"{'='*90}")

# ── BAR CHART + SCORE OVER TIME ──────────────────────────────
latest_signals = data_with_ti.dropna(subset=SIGNAL_COLS).iloc[-1][SIGNAL_COLS].values
indicator_labels = ['RSI','MACD\nHist','BB %','SMA\nCross','EMA\nCross',
                    'ADX','Stoch','ROC','OBV\nTrend','ATR\nRel']

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle(f'Technical Analysis — {SELECTED_NAME}', fontsize=14, fontweight='bold')

colors = ['#2ecc71' if v >= 0.5 else '#e74c3c' for v in latest_signals]
bars = axes[0].bar(indicator_labels, latest_signals, color=colors, edgecolor='white', linewidth=0.8)
axes[0].axhline(0.5, color='gray', linestyle='--', linewidth=1, label='Neutral (0.5)')
axes[0].set_ylim(0, 1.1)
axes[0].set_ylabel('Normalised Strength (0–1)')
axes[0].set_title(f'Indicator Values  |  Score: {score:.1f}/100  |  {signal_label}')
axes[0].legend()
for bar, val in zip(bars, latest_signals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{val:.2f}', ha='center', va='bottom', fontsize=8)

score_series = data_with_ti.dropna(subset=['tech_strength_score']).set_index('date')['tech_strength_score']
axes[1].plot(score_series.index, score_series.values, color='#3498db', linewidth=1)
axes[1].fill_between(score_series.index, 70, score_series.values,
                      where=score_series >= 70, color='#2ecc71', alpha=0.3, label='Bullish (≥70)')
axes[1].fill_between(score_series.index, score_series.values, 30,
                      where=score_series <= 30, color='#e74c3c', alpha=0.3, label='Bearish (≤30)')
axes[1].axhline(70, color='#27ae60', linestyle='--', linewidth=0.8)
axes[1].axhline(30, color='#c0392b', linestyle='--', linewidth=0.8)
axes[1].set_title('Technical Strength Score Over Time')
axes[1].set_ylabel('Score (0–100)')
axes[1].legend()
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.show()
print("✅ Technical analysis complete!")


## CELL 6 — Market Regime Detection on Nifty 50 Index (HMM)

In [ ]:
# ============================================================
# CELL 6: MARKET REGIME DETECTION ON ^NSEI (HMM)
# Uses actual Nifty 50 Index — NOT the selected stock
# ============================================================

def detect_market_regime_nsei(nifty_df, n_regimes=3, lookback=REGIME_LOOKBACK):
    """
    Fits a Gaussian HMM on Nifty 50 Index log-returns + rolling volatility.
    Returns: (regime_df, bull_pct, bear_pct, sideways_pct, hmm_model,
              overall_score, regime_label)
    """
    print("🔍 Detecting Nifty 50 market regimes via HMM...")
    df = nifty_df.copy().sort_values('date')
    df['log_return'] = np.log(df['close'] / df['close'].shift(1))
    df['volatility'] = df['log_return'].rolling(lookback).std().fillna(0)
    df = df.dropna(subset=['log_return'])

    features = df[['log_return', 'volatility']].values

    model = hmm.GaussianHMM(
        n_components=n_regimes, covariance_type='full',
        n_iter=500, random_state=SEED, tol=1e-5)
    model.fit(features)
    hidden_states = model.predict(features)
    df['regime_raw'] = hidden_states

    mean_ret_by_state = {s: df[df['regime_raw'] == s]['log_return'].mean()
                         for s in range(n_regimes)}
    sorted_states = sorted(mean_ret_by_state, key=mean_ret_by_state.get)
    regime_map = {sorted_states[0]: 0, sorted_states[1]: 1, sorted_states[2]: 2}
    df['regime_score'] = df['regime_raw'].map(regime_map)

    counts       = df['regime_score'].value_counts(normalize=True) * 100
    bear_pct     = counts.get(0, 0)
    sideways_pct = counts.get(1, 0)
    bull_pct     = counts.get(2, 0)

    # Overall regime score: 0–100 (0=full bear, 50=sideways, 100=full bull)
    overall_score = (bull_pct * 1.0 + sideways_pct * 0.5 + bear_pct * 0.0)
    if overall_score >= 60:
        regime_label = "🐂 BULL MARKET"
    elif overall_score <= 40:
        regime_label = "🐻 BEAR MARKET"
    else:
        regime_label = "↔ SIDEWAYS MARKET"

    print(f"   Bull:     {bull_pct:.1f}%")
    print(f"   Sideways: {sideways_pct:.1f}%")
    print(f"   Bear:     {bear_pct:.1f}%")
    print(f"   Overall Regime Score : {overall_score:.1f}/100  →  {regime_label}")

    return (df[['date', 'regime_score', 'log_return', 'volatility']],
            bull_pct, bear_pct, sideways_pct, model, overall_score, regime_label)


(regime_df, bull_pct, bear_pct, sideways_pct,
 regime_model, REGIME_SCORE, REGIME_LABEL) = detect_market_regime_nsei(nifty_index, N_REGIMES)

# Merge regime into stock data
data_with_regime = data_with_ti.merge(regime_df[['date', 'regime_score']], on='date', how='left')
data_with_regime['regime_score'] = data_with_regime['regime_score'].fillna(1).astype(int)

# ── Regime Summary ───────────────────────────────────────────
print(f"\n{'='*55}")
print(f"  MARKET REGIME SUMMARY (^NSEI Historical Analysis)")
print(f"{'='*55}")
print(f"  {'Regime':<15} {'% of Time':>10}  {'Score Contribution':>20}")
print(f"  {'-'*50}")
print(f"  {'Bullish':<15} {bull_pct:>9.1f}%  {bull_pct*1.0:>19.1f}")
print(f"  {'Sideways':<15} {sideways_pct:>9.1f}%  {sideways_pct*0.5:>19.1f}")
print(f"  {'Bearish':<15} {bear_pct:>9.1f}%  {bear_pct*0.0:>19.1f}")
print(f"  {'-'*50}")
print(f"  {'Overall Score':<15} {REGIME_SCORE:>9.1f}/100  →  {REGIME_LABEL}")
print(f"{'='*55}")

# ── Plot ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Market Regime Analysis — Nifty 50 Index (^NSEI)', fontsize=14, fontweight='bold')

pie_vals   = [bull_pct, sideways_pct, bear_pct]
pie_labels = [f'Bullish\n{bull_pct:.1f}%', f'Sideways\n{sideways_pct:.1f}%',
              f'Bearish\n{bear_pct:.1f}%']
pie_colors = ['#2ecc71', '#f39c12', '#e74c3c']
wedges, texts = axes[0].pie(pie_vals, labels=pie_labels, colors=pie_colors,
                             startangle=90, wedgeprops=dict(width=0.6))
axes[0].set_title(f'Historical Regime Distribution\nScore: {REGIME_SCORE:.1f}/100  |  {REGIME_LABEL}')

# Regime timeline
nifty_plot = nifty_index.merge(regime_df[['date', 'regime_score']], on='date', how='left')
nifty_plot['regime_score'] = nifty_plot['regime_score'].fillna(1)
regime_colors_map = {0: '#e74c3c', 1: '#f39c12', 2: '#2ecc71'}
regime_names      = {0: 'Bear', 1: 'Sideways', 2: 'Bull'}
for regime_val, rcolor in regime_colors_map.items():
    mask = nifty_plot['regime_score'] == regime_val
    axes[1].fill_between(nifty_plot['date'], nifty_plot['close'],
                          where=mask, alpha=0.35, color=rcolor,
                          label=regime_names[regime_val])
axes[1].plot(nifty_plot['date'], nifty_plot['close'], color='#2c3e50', linewidth=1)
axes[1].set_title('Nifty 50 Price with Regime Overlay')
axes[1].set_ylabel('Nifty 50 Index')
axes[1].legend()
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.show()
print("✅ Market regime detection complete!")


## CELL 7 — Sentiment Analysis (Finnhub + NewsAPI — Real Data Only)

In [ ]:
# ============================================================
# CELL 7: SENTIMENT ANALYSIS — Finnhub + NewsAPI
# Real news data only for the selected company
# NO synthetic / random data used
# ============================================================

vader = SentimentIntensityAnalyzer()

def score_text(text):
    """Returns VADER compound score for a text string."""
    if not text or not isinstance(text, str):
        return None
    vs = vader.polarity_scores(text)
    return vs['compound']  # range: -1.0 to 1.0


def fetch_finnhub_sentiment(symbol, api_key, days=365):
    """Fetches company news headlines from Finnhub and scores with VADER."""    end_dt   = datetime.now()
    start_dt = end_dt - timedelta(days=days)
    url = (f"https://finnhub.io/api/v1/company-news?"
           f"symbol={symbol}&from={start_dt.strftime('%Y-%m-%d')}"
           f"&to={end_dt.strftime('%Y-%m-%d')}&token={api_key}")
    try:
        r = requests.get(url, timeout=12)
        if r.status_code != 200:
            print(f"   Finnhub HTTP {r.status_code}")
            return pd.DataFrame(), []
        news = r.json()
        if not isinstance(news, list) or len(news) == 0:
            print("   Finnhub: 0 articles returned")
            return pd.DataFrame(), []
        records = []
        headlines = []
        for item in news:
            headline = item.get('headline', '') or item.get('summary', '')
            score    = score_text(headline)
            if score is None:
                continue
            dt = pd.to_datetime(item.get('datetime', 0), unit='s')
            records.append({'date': dt.date(), 'sentiment_score': score,
                            'headline': headline, 'source': 'Finnhub'})
            headlines.append({'date': str(dt.date()), 'headline': headline,
                              'sentiment': score, 'source': 'Finnhub'})
        if not records:
            return pd.DataFrame(), []
        df = pd.DataFrame(records)
        df['date'] = pd.to_datetime(df['date'])
        daily = df.groupby('date')['sentiment_score'].mean().reset_index()
        print(f"   Finnhub: {len(records)} headlines → {len(daily)} daily scores")
        return daily, headlines
    except Exception as e:
        print(f"   Finnhub error: {e}")
        return pd.DataFrame(), []


def fetch_newsapi_sentiment(company_name, ticker_base, api_key, days=30):
    """Fetches real news from NewsAPI for the selected company."""    if not NEWSAPI_AVAILABLE or api_key == "YOUR_NEWSAPI_KEY_HERE":
        print("   NewsAPI: Key not configured or library unavailable.")
        return pd.DataFrame(), []
    try:
        newsapi = NewsApiClient(api_key=api_key)
        end_dt   = datetime.now()
        start_dt = end_dt - timedelta(days=days)
        # Search by company name + ticker
        response = newsapi.get_everything(
            q=f'"{company_name}" OR "{ticker_base}"',
            language='en',
            sort_by='publishedAt',
            from_param=start_dt.strftime('%Y-%m-%d'),
            to=end_dt.strftime('%Y-%m-%d'),
            page_size=100
        )
        articles = response.get('articles', [])
        if not articles:
            print("   NewsAPI: 0 articles returned")
            return pd.DataFrame(), []
        records   = []
        headlines = []
        for art in articles:
            title = art.get('title', '') or ''
            desc  = art.get('description', '') or ''
            text  = f"{title}. {desc}".strip()
            score = score_text(text)
            if score is None:
                continue
            pub = art.get('publishedAt', '')[:10]
            try:
                dt = pd.to_datetime(pub)
            except:
                continue
            records.append({'date': dt, 'sentiment_score': score})
            headlines.append({'date': pub, 'headline': title,
                              'sentiment': score, 'source': 'NewsAPI'})
        if not records:
            return pd.DataFrame(), []
        df = pd.DataFrame(records)
        df['date'] = pd.to_datetime(df['date'])
        daily = df.groupby('date')['sentiment_score'].mean().reset_index()
        print(f"   NewsAPI: {len(records)} articles → {len(daily)} daily scores")
        return daily, headlines
    except Exception as e:
        print(f"   NewsAPI error: {e}")
        return pd.DataFrame(), []


def build_sentiment_column(stock_df, selected_ticker, selected_name,
                            finnhub_key, newsapi_key):
    """
    Merges Finnhub + NewsAPI sentiment for the SELECTED company only.
    No synthetic / random fallback — missing dates get NaN then forward-filled.
    """
    ticker_base = selected_ticker.replace('.NS', '')
    print(f"\n📰 Fetching real sentiment data for {selected_name} ({ticker_base})...")

    fh_df,  fh_headlines  = fetch_finnhub_sentiment(FINNHUB_SYMBOL, finnhub_key, days=365)
    na_df,  na_headlines  = fetch_newsapi_sentiment(selected_name, ticker_base,
                                                     newsapi_key, days=30)

    # Combine all real headlines for top-5 display
    all_headlines = fh_headlines + na_headlines
    all_headlines = sorted(all_headlines, key=lambda x: x['date'], reverse=True)

    sentiment_parts = [p for p in [fh_df, na_df] if not p.empty]
    if sentiment_parts:
        combined = pd.concat(sentiment_parts).groupby('date')['sentiment_score'].mean().reset_index()
        print(f"   Combined: {len(combined)} daily sentiment data points (real data only)")
    else:
        print("   ⚠  No API sentiment data retrieved. sentiment_score will be 0.")
        combined = pd.DataFrame()

    result = stock_df.copy()
    if not combined.empty:
        combined['date'] = pd.to_datetime(combined['date'])
        result = result.merge(combined, on='date', how='left')
        # Forward-fill missing dates (weekend gaps etc.) — NO random fill
        result['sentiment_score'] = result['sentiment_score'].ffill().bfill().fillna(0)
    else:
        result['sentiment_score'] = 0.0

    result['sentiment_valid'] = (result['sentiment_score'].abs() > 0.05).astype(int)
    print(f"   Valid sentiment rows: {result['sentiment_valid'].sum()}/{len(result)}")

    # Compute optimised (exponentially-smoothed) sentiment
    result['sentiment_smooth'] = result['sentiment_score'].ewm(span=7, adjust=False).mean()

    latest_raw    = result['sentiment_score'].iloc[-1]
    latest_smooth = result['sentiment_smooth'].iloc[-1]
    print(f"\n   Latest raw sentiment     : {latest_raw:+.4f}")
    print(f"   Latest smoothed sentiment: {latest_smooth:+.4f}  ← used in model")

    return result, all_headlines


data_with_sentiment, SENTIMENT_HEADLINES = build_sentiment_column(
    data_with_regime, SELECTED_TICKER, SELECTED_NAME,
    FINNHUB_API_KEY, NEWSAPI_KEY)

# Replace sentiment_score in features with smoothed version
data_with_sentiment['sentiment_score'] = data_with_sentiment['sentiment_smooth']

# ── Sentiment Score Summary ──────────────────────────────────
latest_sent_val = float(data_with_sentiment['sentiment_score'].iloc[-1])
sent_label = ("BULLISH" if latest_sent_val > 0.1 else
             ("BEARISH" if latest_sent_val < -0.1 else "NEUTRAL"))
print(f"\n{'='*50}")
print(f"  OPTIMISED SENTIMENT SCORE : {latest_sent_val:+.4f}")
print(f"  Sentiment Label           : {sent_label}")
print(f"{'='*50}")

# ── Sentiment Chart ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 5))
recent = data_with_sentiment[data_with_sentiment['date'] >= '2023-01-01'].copy()
if len(recent) > 0 and recent['sentiment_score'].abs().sum() > 0:
    ax.plot(recent['date'], recent['sentiment_score'], alpha=0.5, color='#3498db',
            linewidth=0.8, label='Smoothed Sentiment Score')
    rolling_sent = recent['sentiment_score'].rolling(20).mean()
    ax.plot(recent['date'], rolling_sent, color='#2c3e50', linewidth=2,
            label='20-day Rolling Mean')
    ax.fill_between(recent['date'], rolling_sent.fillna(0), 0,
                    where=rolling_sent >= 0, alpha=0.25, color='#27ae60', label='Positive Zone')
    ax.fill_between(recent['date'], rolling_sent.fillna(0), 0,
                    where=rolling_sent < 0,  alpha=0.25, color='#e74c3c', label='Negative Zone')
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_title(f'Sentiment Analysis — {SELECTED_NAME} (2023–Present)', fontsize=13, fontweight='bold')
    ax.set_ylabel('Sentiment Score (−1 Bearish → +1 Bullish)')
    ax.legend(loc='upper left')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.set_ylim(-1.1, 1.1)
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()
    print("✅ Sentiment chart rendered!")
else:
    print("⚠  Sentiment data is zero (API keys may not be configured). Chart skipped.")
    plt.close(fig)

# ── Top 5 News Headlines ─────────────────────────────────────
print(f"\n{'='*70}")
print(f"  TOP 5 RECENT NEWS — {SELECTED_NAME}")
print(f"{'='*70}")
if SENTIMENT_HEADLINES:
    for i, art in enumerate(SENTIMENT_HEADLINES[:5], 1):
        sent_str = f"{art['sentiment']:+.3f}"
        tone = "🟢" if art['sentiment'] > 0.05 else ("🔴" if art['sentiment'] < -0.05 else "🟡")
        print(f"  {i}. [{art['date']}] {tone} Sentiment: {sent_str}")
        print(f"     {art['headline'][:100]}")
        print(f"     Source: {art['source']}")
        print()
else:
    print("  No headlines available (API keys not configured).")
print(f"{'='*70}")
print("✅ Sentiment analysis complete!")


## CELL 8 — Data Cleaning & Feature Preparation (All 10 Stocks for Training)

In [ ]:
# ============================================================
# CELL 8: DATA CLEANING & FEATURE PREPARATION
# PPO trains on ALL 10 stocks merged; eval uses selected stock
# ============================================================

FEATURE_COLS = [
    'close_norm', 'daily_return', 'log_return', 'volatility_10',
    'macd', 'macd_signal', 'macd_hist',
    'sma_10', 'sma_50', 'ema_20',
    'rsi', 'stoch_k', 'stoch_d', 'roc',
    'bb_pct', 'bb_width', 'atr',
    'obv',
    'adx',
    'sentiment_score',
    'regime_score',
    'tech_strength_score',
]


def prepare_single_stock(df, feature_cols, scaler=None, fit=False):
    """Clean and scale features for one stock's DataFrame."""
    df = df.copy()
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df[feature_cols] = df[feature_cols].ffill().bfill().fillna(0)
    if fit:
        scaler = StandardScaler()
        df[feature_cols] = scaler.fit_transform(df[feature_cols])
    else:
        df[feature_cols] = scaler.transform(df[feature_cols])
    return df, scaler


# ── Process ALL 10 stocks for TRAINING ───────────────────────
print("🔧 Processing all 10 stocks for PPO training...")
all_train_dfs = []
all_val_dfs   = []

# Fit scaler on selected stock's train set; apply to all
sel_train_raw = data_with_sentiment[
    (data_with_sentiment['date'] >= TRAIN_START) &
    (data_with_sentiment['date'] <= TRAIN_END)].copy()
_, feature_scaler = prepare_single_stock(sel_train_raw, FEATURE_COLS, fit=True)

for ticker, name in ALL_TICKERS:
    if ticker not in all_raw_data:
        continue
    raw  = all_raw_data[ticker]
    ti   = add_technical_indicators(raw)
    ti, _ = compute_technical_strength_score(ti)

    # Merge regime
    ti = ti.merge(regime_df[['date', 'regime_score']], on='date', how='left')
    ti['regime_score'] = ti['regime_score'].fillna(1).astype(int)

    # Sentiment: use selected stock sentiment for all (regime generalisation)
    ti = ti.merge(data_with_sentiment[['date', 'sentiment_score']],
                  on='date', how='left')
    ti['sentiment_score'] = ti['sentiment_score'].ffill().bfill().fillna(0)
    ti['ticker'] = ticker

    ti_clean, _ = prepare_single_stock(ti, FEATURE_COLS, scaler=feature_scaler)

    tr = ti_clean[(ti_clean['date'] >= TRAIN_START) & (ti_clean['date'] <= TRAIN_END)]
    vl = ti_clean[(ti_clean['date'] >= VAL_START)   & (ti_clean['date'] <= VAL_END)]

    if len(tr) > 0:
        all_train_dfs.append(tr)
    if len(vl) > 0:
        all_val_dfs.append(vl)
    print(f"  ✅ {ticker:<20} train={len(tr):>4}  val={len(vl):>4}")

combined_train = pd.concat(all_train_dfs, ignore_index=True).sort_values('date').reset_index(drop=True)
combined_val   = pd.concat(all_val_dfs,   ignore_index=True).sort_values('date').reset_index(drop=True)

# ── Selected stock only for TEST ─────────────────────────────
clean_selected, _ = prepare_single_stock(data_with_sentiment, FEATURE_COLS,
                                          scaler=feature_scaler)
test_data = clean_selected[(clean_selected['date'] >= TEST_START) &
                            (clean_selected['date'] <= TEST_END)].copy()
val_data  = clean_selected[(clean_selected['date'] >= VAL_START)  &
                            (clean_selected['date'] <= VAL_END)].copy()
train_data = clean_selected[(clean_selected['date'] >= TRAIN_START) &
                             (clean_selected['date'] <= TRAIN_END)].copy()

FINAL_FEATURES = FEATURE_COLS
N_FEATURES     = len(FINAL_FEATURES)

print(f"\n{'='*55}")
print(f"  TRAINING DATA — All 10 Stocks Combined")
print(f"{'='*55}")
print(f"  Train rows (combined) : {len(combined_train):,}")
print(f"  Val   rows (combined) : {len(combined_val):,}")
print(f"  Test  rows (selected) : {len(test_data):,}  [{SELECTED_TICKER}]")
print(f"  Features in state     : {N_FEATURES}")
print(f"{'='*55}")
print("✅ Data preparation complete!")


## CELL 9 — Custom RL Trading Environment

In [ ]:
# ============================================================
# CELL 9: CUSTOM RL TRADING ENVIRONMENT
# Works on any single-stock DataFrame slice
# ============================================================

class NiftyRecommendationEnv(gym.Env):
    """
    Single-stock trading environment for BUY/HOLD/SELL recommendation.
    State: [position (1), features (N_FEATURES), price_norm (1)]
    Action: scalar in [-1, 1]  (< -0.3 = SELL, > 0.3 = BUY, else HOLD)
    Reward: Sharpe-adjusted daily return − transaction cost
    """

    metadata = {'render_modes': ['human']}

    def __init__(self, df, feature_cols,
                 dummy_capital=DUMMY_CAPITAL,
                 transaction_cost=TRANSACTION_COST,
                 max_shares=MAX_SHARES,
                 sharpe_window=20):
        super().__init__()
        self.df               = df.reset_index(drop=True)
        self.feature_cols     = feature_cols
        self.n_features       = len(feature_cols)
        self.dummy_capital    = dummy_capital
        self.transaction_cost = transaction_cost
        self.max_shares       = max_shares
        self.sharpe_window    = sharpe_window
        self.dates            = sorted(df['date'].unique())
        self.n_days           = len(self.dates)

        obs_size = 1 + self.n_features + 1
        self.action_space      = spaces.Box(low=-1, high=1, shape=(1,), dtype=np.float32)
        self.observation_space = spaces.Box(low=-10, high=10, shape=(obs_size,), dtype=np.float32)
        self.reset()

    def _get_row(self, step):
        date = self.dates[step]
        rows = self.df[self.df['date'] == date]
        return rows.iloc[0] if len(rows) > 0 else None

    def _obs(self):
        row = self._get_row(self.current_step)
        if row is None:
            return np.zeros(self.observation_space.shape, dtype=np.float32)
        feats      = row[self.feature_cols].values.astype(np.float32)
        price_norm = np.array([row['close_norm']], dtype=np.float32)
        position   = np.array([self.position], dtype=np.float32)
        obs        = np.concatenate([position, feats, price_norm])
        obs        = np.nan_to_num(obs, nan=0.0, posinf=5.0, neginf=-5.0)
        return obs.astype(np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step   = 0
        self.balance        = float(self.dummy_capital)
        self.shares_held    = 0
        self.position       = 0.0
        self.portfolio_hist = [self.dummy_capital]
        self.returns_hist   = []
        self.actions_hist   = []
        return self._obs(), {}

    def _portfolio_value(self):
        row   = self._get_row(self.current_step)
        price = float(row['close']) if row is not None else 0.0
        return self.balance + self.shares_held * price

    def step(self, action):
        action_scalar = float(action[0])
        self.actions_hist.append(action_scalar)

        row   = self._get_row(self.current_step)
        price = float(row['close']) if row is not None else 1.0

        if action_scalar > 0.3:
            n_shares = int((self.balance * 0.3) / (price + 1e-8))
            n_shares = min(n_shares, self.max_shares)
            if n_shares > 0 and self.balance >= n_shares * price * (1 + self.transaction_cost):
                self.balance    -= n_shares * price * (1 + self.transaction_cost)
                self.shares_held += n_shares
                self.position    = min(1.0, self.position + 0.3)
        elif action_scalar < -0.3:
            n_sell = min(self.shares_held, max(1, int(self.shares_held * 0.5)))
            if n_sell > 0:
                self.balance    += n_sell * price * (1 - self.transaction_cost)
                self.shares_held -= n_sell
                self.position    = max(-1.0, self.position - 0.3)

        self.current_step += 1
        done = self.current_step >= self.n_days - 1

        new_val  = self._portfolio_value()
        prev_val = self.portfolio_hist[-1]
        pct_ret  = (new_val - prev_val) / (prev_val + 1e-8)
        self.portfolio_hist.append(new_val)
        self.returns_hist.append(pct_ret)

        if len(self.returns_hist) >= self.sharpe_window:
            r_arr  = np.array(self.returns_hist[-self.sharpe_window:])
            sharpe = r_arr.mean() / (r_arr.std() + 1e-8)
            peak   = max(self.portfolio_hist)
            dd_pen = max(0, (peak - new_val) / (peak + 1e-8)) * 0.5
            reward = (sharpe - dd_pen) * 10
        else:
            reward = pct_ret * 100

        obs  = self._obs()
        info = {'portfolio_value': new_val, 'daily_return': pct_ret, 'action': action_scalar}
        return obs, float(reward), done, False, info

    def render(self):
        pv  = self._portfolio_value()
        ret = (pv / self.dummy_capital - 1) * 100
        print(f"  Step {self.current_step:>4}: Portfolio=₹{pv:,.0f} ({ret:+.2f}%)")


# Sanity check
print("🧪 Environment sanity check...")
_env = NiftyRecommendationEnv(combined_train, FINAL_FEATURES)
obs0, _ = _env.reset()
print(f"   Observation shape : {obs0.shape}")
print(f"   Action space      : {_env.action_space}")
print(f"   Trading days (combined train): {_env.n_days}")
a0   = _env.action_space.sample()
obs1, rew, done, _, info1 = _env.step(a0)
print(f"   Sample reward     : {rew:.4f}")
print(f"   Portfolio value   : ₹{info1['portfolio_value']:,.0f}")
print("✅ Environment ready!")


## CELL 10 — Train PPO Agent on All 10 Stocks

In [ ]:
# ============================================================
# CELL 10: TRAIN PPO ON ALL 10 STOCKS
# Training env uses combined_train (all tickers)
# Val env uses combined_val (all tickers)
# ============================================================

import os
os.makedirs('trained_models', exist_ok=True)
os.makedirs('logs', exist_ok=True)

def make_train_env():
    env = NiftyRecommendationEnv(
        df=combined_train,
        feature_cols=FINAL_FEATURES,
        transaction_cost=TRANSACTION_COST,
        max_shares=MAX_SHARES,
        sharpe_window=20)
    return Monitor(env)

def make_val_env():
    env = NiftyRecommendationEnv(
        df=combined_val,
        feature_cols=FINAL_FEATURES,
        transaction_cost=TRANSACTION_COST,
        max_shares=MAX_SHARES,
        sharpe_window=20)
    return Monitor(env)


train_vec = DummyVecEnv([make_train_env])
val_vec   = DummyVecEnv([make_val_env])

ppo_model = PPO(
    policy        = "MlpPolicy",
    env           = train_vec,
    learning_rate = LEARNING_RATE,
    n_steps       = N_STEPS,
    batch_size    = BATCH_SIZE,
    n_epochs      = N_EPOCHS,
    gamma         = GAMMA,
    gae_lambda    = GAE_LAMBDA,
    clip_range    = CLIP_RANGE,
    ent_coef      = ENT_COEF,
    vf_coef       = VF_COEF,
    max_grad_norm = MAX_GRAD_NORM,
    policy_kwargs = dict(net_arch=[dict(pi=[256, 256, 128], vf=[256, 256, 128])]),
    verbose       = 1,
    seed          = SEED,
    tensorboard_log = './logs'
)

checkpoint_cb = CheckpointCallback(
    save_freq=25_000, save_path='./trained_models/', name_prefix='ppo_nifty')
eval_cb = EvalCallback(
    val_vec, best_model_save_path='./trained_models/',
    log_path='./logs/', eval_freq=10_000, n_eval_episodes=1,
    deterministic=True, verbose=0)

print(f"🤖 Training PPO on ALL 10 Nifty stocks for {TOTAL_TIMESTEPS:,} timesteps...")
print(f"   Hyperparameters:")
print(f"   LR={LEARNING_RATE}  Steps={N_STEPS}  Batch={BATCH_SIZE}  Epochs={N_EPOCHS}")
print(f"   Gamma={GAMMA}  GaeLambda={GAE_LAMBDA}  ClipRange={CLIP_RANGE}")
print(f"   EntCoef={ENT_COEF}  VfCoef={VF_COEF}  MaxGradNorm={MAX_GRAD_NORM}")
print(f"   Net: [256, 256, 128]")
print(f"   ⏳ Training in progress...")

ppo_model.learn(
    total_timesteps = TOTAL_TIMESTEPS,
    callback        = [checkpoint_cb, eval_cb],
    progress_bar    = True
)

model_path = f'trained_models/PPO_nifty_all10'
ppo_model.save(model_path)
print(f"\n✅ PPO model trained on all 10 stocks → saved: {model_path}.zip")
print(f"   Evaluation will use: {SELECTED_TICKER} (selected stock only)")


## CELL 11 — Backtesting (Selected Stock — Validation + Test)

In [ ]:
# ============================================================
# CELL 11: BACKTESTING — SELECTED STOCK ONLY
# Load best model, run on val_data + test_data (selected stock)
# ============================================================

def run_backtest(model, df, label="Test"):
    """Runs trained model deterministically on a split."""
    print(f"\n📊 Running backtest: {label} ({len(df)} rows — {SELECTED_TICKER})")
    env  = NiftyRecommendationEnv(df=df, feature_cols=FINAL_FEATURES,
                                   transaction_cost=TRANSACTION_COST,
                                   max_shares=MAX_SHARES, sharpe_window=20)
    obs, _ = env.reset()
    done   = False
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, _, done, _, _ = env.step(action)
    return env.portfolio_hist, env.returns_hist, env.actions_hist


def buy_and_hold(df, dummy_capital):
    df_s = df.sort_values('date').reset_index(drop=True)
    if df_s.empty:
        return [dummy_capital]
    shares = dummy_capital / (df_s['close'].iloc[0] + 1e-8)
    return (df_s['close'].values * shares).tolist()


# Load best model saved by EvalCallback (if exists)
best_path = 'trained_models/best_model'
if os.path.exists(best_path + '.zip'):
    print("📂 Loading best validation model...")
    ppo_model = PPO.load(best_path)
else:
    print("ℹ  Using most recent trained model (best_model not found).")

val_hist,  val_rets,  val_acts  = run_backtest(ppo_model, val_data,  "Validation (2024)")
test_hist, test_rets, test_acts = run_backtest(ppo_model, test_data, "Test (2025–present)")

bnh_val  = buy_and_hold(val_data,  DUMMY_CAPITAL)
bnh_test = buy_and_hold(test_data, DUMMY_CAPITAL)

print("\n✅ Backtesting complete!")


## CELL 12 — Evaluation Metrics

In [ ]:
# ============================================================
# CELL 12: EVALUATION METRICS
# ============================================================

def compute_metrics(portfolio_hist, returns_hist, actions_hist=None, label=""):
    vals   = np.array(portfolio_hist)
    rets   = np.array(returns_hist) if returns_hist else np.diff(vals) / (vals[:-1] + 1e-8)
    init   = vals[0];  final = vals[-1];  n_days = len(vals)

    cum_ret = (final / init - 1) * 100
    ann_ret = ((final / init) ** (252 / n_days) - 1) * 100
    sharpe  = (rets.mean() / (rets.std() + 1e-8)) * np.sqrt(252)
    neg_rets = rets[rets < 0]
    sortino  = (rets.mean() / (neg_rets.std() + 1e-8)) * np.sqrt(252)
    peak   = np.maximum.accumulate(vals)
    dd     = (vals - peak) / (peak + 1e-8)
    max_dd = dd.min() * 100
    calmar = ann_ret / (abs(max_dd) + 1e-8)
    win_rate = (rets > 0).mean() * 100
    ann_vol  = rets.std() * np.sqrt(252) * 100

    precision_val = None
    sell_precision = None
    if actions_hist and len(actions_hist) == len(rets):
        buy_mask  = np.array(actions_hist) > 0.3
        sell_mask = np.array(actions_hist) < -0.3
        if buy_mask.sum() > 0:
            precision_val = ((buy_mask & (rets > 0)).sum() / buy_mask.sum()) * 100
        if sell_mask.sum() > 0:
            sell_precision = ((sell_mask & (rets < 0)).sum() / sell_mask.sum()) * 100

    return {
        'Label'               : label,
        'Cumulative Return'   : f"{cum_ret:+.2f}%",
        'Annualised Return'   : f"{ann_ret:+.2f}%",
        'Annualised Volatility': f"{ann_vol:.2f}%",
        'Sharpe Ratio'        : f"{sharpe:.4f}",
        'Sortino Ratio'       : f"{sortino:.4f}",
        'Calmar Ratio'        : f"{calmar:.4f}",
        'Max Drawdown'        : f"{max_dd:.2f}%",
        'Win Rate'            : f"{win_rate:.2f}%",
        'Buy Precision'       : f"{precision_val:.2f}%" if precision_val else "N/A",
        'Sell Precision'      : f"{sell_precision:.2f}%" if sell_precision else "N/A",
        '_cum_ret' : cum_ret, '_sharpe': sharpe, '_max_dd': max_dd,
        '_win_rate': win_rate, '_sortino': sortino, '_calmar': calmar,
    }


BENCHMARKS = {
    'Sharpe Ratio'         : ('> 1.0', 'Good | > 2.0 Excellent'),
    'Sortino Ratio'        : ('> 1.5', 'Good | > 3.0 Excellent'),
    'Calmar Ratio'         : ('> 0.5', 'Good | > 1.0 Excellent'),
    'Max Drawdown'         : ('> -20%', 'Acceptable | < -10% Preferred'),
    'Win Rate'             : ('> 50%', 'Acceptable | > 55% Preferred'),
    'Buy Precision'        : ('> 50%', 'Acceptable | > 60% Preferred'),
    'Cumulative Return'    : ('> 0%', 'Positive return required'),
    'Annualised Return'    : ('> 12%', 'Beats typical FD/index return'),
}

val_metrics  = compute_metrics(val_hist,  val_rets,  val_acts,  f"PPO (Val 2024)")
test_metrics = compute_metrics(test_hist, test_rets, test_acts, f"PPO (Test 2025–{datetime.now().year})")
bnh_val_m    = compute_metrics(bnh_val,   [], None, "B&H (Val)")
bnh_test_m   = compute_metrics(bnh_test,  [], None, "B&H (Test)")

display_keys = ['Cumulative Return','Annualised Return','Annualised Volatility',
                'Sharpe Ratio','Sortino Ratio','Calmar Ratio',
                'Max Drawdown','Win Rate','Buy Precision','Sell Precision']

print("\n" + "="*90)
print(f"    PERFORMANCE METRICS — {SELECTED_NAME} ({SELECTED_TICKER})")
print("="*90)
print(f"{'Metric':<28} {'PPO Val':>12} {'B&H Val':>12} {'PPO Test':>12} {'B&H Test':>12}  {'Benchmark'}")
print("-"*90)
for key in display_keys:
    bench_str = BENCHMARKS.get(key, ('-',''))[0]
    print(f"{key:<28} {val_metrics.get(key,'N/A'):>12} {bnh_val_m.get(key,'N/A'):>12} "
          f"{test_metrics.get(key,'N/A'):>12} {bnh_test_m.get(key,'N/A'):>12}  {bench_str}")
print("="*90)


## CELL 13 — Performance Visualisation

In [ ]:
# ============================================================
# CELL 13: PERFORMANCE VISUALISATION
# ============================================================

sns.set_theme(style='darkgrid', palette='muted')

test_dates = sorted(test_data['date'].unique())
val_dates  = sorted(val_data['date'].unique())

# ── 1. Portfolio Value ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 5))
n_test  = min(len(test_hist), len(test_dates))
n_bnh_t = min(len(bnh_test),  len(test_dates))
ax.plot(test_dates[:n_test],
        [v / DUMMY_CAPITAL * 100 for v in test_hist[:n_test]],
        label='PPO Agent', color='#2980b9', linewidth=2)
ax.plot(test_dates[:n_bnh_t],
        [v / DUMMY_CAPITAL * 100 for v in bnh_test[:n_bnh_t]],
        label='Buy & Hold', color='#e67e22', linewidth=2, linestyle='--')
ax.axhline(100, color='gray', linestyle=':', linewidth=0.8)
ax.set_title(f'Portfolio Performance (Test 2025–{datetime.now().year}) — {SELECTED_NAME}',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Portfolio Value (% of initial)')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=30);  plt.tight_layout();  plt.show()

# ── 2. Drawdown ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 4))
pv_arr = np.array(test_hist[:n_test])
peak   = np.maximum.accumulate(pv_arr)
dd_arr = (pv_arr - peak) / (peak + 1e-8) * 100
ax.fill_between(test_dates[:n_test], dd_arr, 0, alpha=0.6, color='#e74c3c')
ax.plot(test_dates[:n_test], dd_arr, color='#c0392b', linewidth=1)
ax.set_title('Portfolio Drawdown (Test Period)', fontsize=12, fontweight='bold')
ax.set_ylabel('Drawdown (%)');  ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=30);  plt.tight_layout();  plt.show()

# ── 3. Daily Returns Distribution ────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
ret_arr = np.array(test_rets) * 100
ax.hist(ret_arr, bins=60, color='#3498db', alpha=0.75, edgecolor='white')
ax.axvline(0, color='black', linewidth=1.2, linestyle='--')
ax.axvline(ret_arr.mean(), color='#2ecc71', linewidth=1.5,
           label=f'Mean: {ret_arr.mean():.3f}%')
ax.axvline(np.percentile(ret_arr, 5), color='#e74c3c', linewidth=1.5, linestyle=':',
           label=f'5th pct: {np.percentile(ret_arr,5):.3f}%')
ax.set_title('Daily Returns Distribution (Test Period)', fontsize=12, fontweight='bold')
ax.set_xlabel('Daily Return (%)');  ax.set_ylabel('Frequency')
ax.legend();  plt.tight_layout();  plt.show()

# ── 4. Key Metrics Bar Chart ──────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(f'Key Metrics: PPO vs Buy & Hold — {SELECTED_NAME}', fontsize=13, fontweight='bold')
for (title, key, ax) in [
    ('Sharpe Ratio', '_sharpe', axes[0]),
    ('Cumulative Return (%)', '_cum_ret', axes[1]),
    ('Max Drawdown (%)', '_max_dd', axes[2])
]:
    vals_  = [test_metrics[key], bnh_test_m[key]]
    bars   = ax.bar(['PPO','B&H'], vals_, color=['#2980b9','#e67e22'], alpha=0.85, edgecolor='white')
    for bar, v in zip(bars, vals_):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + abs(bar.get_height()) * 0.02,
                f'{v:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.set_title(title);  ax.axhline(0, color='black', linewidth=0.8)
plt.tight_layout();  plt.show()

# ── 5. Action Distribution ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('PPO Agent Action Analysis (Test Period)', fontsize=12, fontweight='bold')
acts = np.array(test_acts)
axes[0].hist(acts, bins=50, color='#9b59b6', alpha=0.8, edgecolor='white')
axes[0].axvline(0.3,  color='#27ae60', linewidth=1.5, linestyle='--', label='Buy threshold')
axes[0].axvline(-0.3, color='#e74c3c', linewidth=1.5, linestyle='--', label='Sell threshold')
axes[0].set_title('Action Value Distribution')
axes[0].set_xlabel('Action (−1=Sell, 0=Hold, +1=Buy)');  axes[0].legend()

buy_count  = (acts >  0.3).sum()
sell_count = (acts < -0.3).sum()
hold_count = len(acts) - buy_count - sell_count
bars2 = axes[1].bar(['BUY','HOLD','SELL'], [buy_count, hold_count, sell_count],
                     color=['#27ae60','#f39c12','#e74c3c'], alpha=0.85, edgecolor='white')
axes[1].set_title('BUY / HOLD / SELL Count');  axes[1].set_ylabel('Number of Days')
for i, v in enumerate([buy_count, hold_count, sell_count]):
    axes[1].text(i, v + 0.5, str(v), ha='center', va='bottom', fontweight='bold')
plt.tight_layout();  plt.show()

# ── 6. Rolling 30-day Sharpe ──────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 4))
rets_s = pd.Series(test_rets)
rolling_sharpe = rets_s.rolling(30).apply(
    lambda x: (x.mean() / (x.std() + 1e-8)) * np.sqrt(252), raw=True)
ax.plot(test_dates[:len(rolling_sharpe)], rolling_sharpe.values, color='#1abc9c', linewidth=1.5)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.axhline(1, color='#27ae60', linewidth=0.8, linestyle=':', label='Sharpe=1')
ax.fill_between(test_dates[:len(rolling_sharpe)], rolling_sharpe.fillna(0), 0,
                where=rolling_sharpe >= 0, alpha=0.25, color='#27ae60')
ax.fill_between(test_dates[:len(rolling_sharpe)], rolling_sharpe.fillna(0), 0,
                where=rolling_sharpe < 0,  alpha=0.25, color='#e74c3c')
ax.set_title('Rolling 30-Day Sharpe Ratio (Test Period)', fontsize=12, fontweight='bold')
ax.set_ylabel('Rolling Sharpe');  ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=30);  plt.tight_layout();  plt.show()

print("✅ All visualisations rendered!")


## CELL 14 — Final Recommendation Output

In [ ]:
# ============================================================
# CELL 14: FINAL RECOMMENDATION OUTPUT
# Prints: Technical Score | Sentiment Score | Regime Score
# Then: PPO Final Signal (no confidence shown)
# Then: Top 5 News headlines
# ============================================================

def generate_recommendation(model, test_df, latest_sentiment,
                             tech_score, bull_pct, bear_pct, sideways_pct):
    env = NiftyRecommendationEnv(df=test_df, feature_cols=FINAL_FEATURES,
                                  transaction_cost=0, max_shares=MAX_SHARES)
    obs, _ = env.reset()
    done   = False
    last_action = 0.0
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, _, done, _, _ = env.step(action)
        last_action = float(action[0])

    # PPO signal
    if last_action > 0.3:
        ppo_signal = 'BUY'
    elif last_action < -0.3:
        ppo_signal = 'SELL'
    else:
        ppo_signal = 'HOLD'

    # Technical signal
    if tech_score >= 65:
        tech_signal = 'BUY'
    elif tech_score <= 35:
        tech_signal = 'SELL'
    else:
        tech_signal = 'HOLD'

    # Sentiment signal
    if abs(latest_sentiment) < 0.05:
        sent_signal = 'NEUTRAL'
    elif latest_sentiment > 0.1:
        sent_signal = 'BUY'
    elif latest_sentiment < -0.1:
        sent_signal = 'SELL'
    else:
        sent_signal = 'HOLD'

    # Regime signal
    dominant = ('Bull' if bull_pct > max(bear_pct, sideways_pct) else
               ('Bear' if bear_pct > max(bull_pct, sideways_pct) else 'Sideways'))
    regime_signal = 'BUY' if dominant == 'Bull' else ('SELL' if dominant == 'Bear' else 'HOLD')

    # Weighted vote (PPO highest weight)
    score_map = {'BUY': 1, 'HOLD': 0, 'SELL': -1, 'NEUTRAL': 0}
    weighted  = (score_map[ppo_signal]    * 0.50 +
                 score_map[tech_signal]   * 0.25 +
                 score_map[sent_signal]   * 0.15 +
                 score_map[regime_signal] * 0.10)

    if weighted > 0.15:
        final = 'BUY'
    elif weighted < -0.15:
        final = 'SELL'
    else:
        final = 'HOLD'

    return {
        'ppo_signal'    : ppo_signal,
        'ppo_raw_action': last_action,
        'tech_signal'   : tech_signal,
        'tech_score'    : tech_score,
        'sent_signal'   : sent_signal,
        'sent_score'    : latest_sentiment,
        'regime_signal' : regime_signal,
        'regime_dominant': dominant,
        'weighted_score': weighted,
        'final'         : final,
    }


# Get latest values
latest_sent_val   = float(data_with_sentiment['sentiment_score'].iloc[-1])
latest_tech_score = float(data_with_ti.dropna(subset=['tech_strength_score'])['tech_strength_score'].iloc[-1])

rec = generate_recommendation(
    ppo_model, test_data, latest_sent_val,
    latest_tech_score, bull_pct, bear_pct, sideways_pct)

# ── Colour helpers ───────────────────────────────────────────
SIG_COLOR = {'BUY': '\033[92m', 'SELL': '\033[91m', 'HOLD': '\033[93m',
             'NEUTRAL': '\033[90m', 'BULL MARKET': '\033[92m',
             'BEAR MARKET': '\033[91m', 'SIDEWAYS MARKET': '\033[93m'}
RESET = '\033[0m'

def csig(sig):
    return f"{SIG_COLOR.get(sig, '')}{sig}{RESET}"

def score_bar(score, width=20):
    filled = int(score / 100 * width)
    return '[' + '█' * filled + '░' * (width - filled) + f'] {score:.1f}/100'

# Sentiment label
sent_label_str = ("BULLISH" if rec['sent_score'] > 0.1 else
                  ("BEARISH" if rec['sent_score'] < -0.1 else "NEUTRAL"))

# ── FINAL OUTPUT TABLE ────────────────────────────────────────
print("\n" + "="*70)
print(f"  🔮  FINAL RECOMMENDATION — {SELECTED_NAME} ({SELECTED_TICKER})")
print(f"       As of: {datetime.now().strftime('%Y-%m-%d')}")
print("="*70)
print(f"  {'Metric':<30} {'Value':>20}  {'Signal'}")
print(f"  {'-'*68}")
print(f"  {'Technical Score':<30} {score_bar(rec['tech_score']):>20}  {csig(rec['tech_signal'])}")
print(f"  {'Sentiment Score':<30} {rec['sent_score']:>+20.4f}  {csig(sent_label_str)}")
print(f"  {'Market Regime Score':<30} {score_bar(REGIME_SCORE):>20}  {csig(REGIME_LABEL.replace('🐂 ','').replace('🐻 ','').replace('↔ ',''))}")
print(f"  {'-'*68}")
print(f"  {'PPO Agent (Final Signal)':<30} {'':>20}  {csig(rec['ppo_signal'])}  (raw: {rec['ppo_raw_action']:+.4f})")
print(f"  {'Weighted Score':<30} {rec['weighted_score']:>+20.4f}")
print(f"  {'-'*68}")
print(f"  ▶  FINAL CALL   :  {csig(rec['final'])}")
print("="*70)

# ── Recommendation visual bar chart ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Final Recommendation Summary — {SELECTED_NAME}', fontsize=13, fontweight='bold')

scores_for_plot = [rec['tech_score'], (rec['sent_score'] + 1) * 50, REGIME_SCORE]
labels_for_plot = ['Technical\nScore', 'Sentiment\nScore', 'Regime\nScore']
colors_plot     = ['#3498db', '#9b59b6', '#e67e22']
bars_r = axes[0].bar(labels_for_plot, scores_for_plot, color=colors_plot, alpha=0.85, edgecolor='white')
axes[0].set_ylim(0, 110)
axes[0].axhline(50, color='gray', linestyle='--', linewidth=1, label='Neutral (50)')
axes[0].set_ylabel('Score (0–100)')
axes[0].set_title('Component Scores')
axes[0].legend()
for bar, v in zip(bars_r, scores_for_plot):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
                 f'{v:.1f}', ha='center', va='bottom', fontweight='bold')

signal_map = {'BUY': 1, 'HOLD': 0, 'SELL': -1, 'NEUTRAL': 0}
signal_scores = [signal_map[rec['ppo_signal']],
                 signal_map[rec['tech_signal']],
                 signal_map[sent_label_str],
                 signal_map[rec['regime_signal']]]
sig_labels = ['PPO\nAgent', 'Technical', 'Sentiment', 'Regime']
sig_colors = ['#2ecc71' if s > 0 else ('#e74c3c' if s < 0 else '#f39c12')
              for s in signal_scores]
axes[1].bar(sig_labels, signal_scores, color=sig_colors, alpha=0.85, edgecolor='white')
axes[1].set_ylim(-1.5, 1.5)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_ylabel('Signal (−1=Sell, 0=Hold, +1=Buy)')
axes[1].set_title(f'Component Signals  |  Final: {rec["final"]}')
for i, (v, lbl) in enumerate(zip(signal_scores, ['BUY' if s>0 else ('SELL' if s<0 else 'HOLD')
                                                   for s in signal_scores])):
    axes[1].text(i, v + (0.07 if v >= 0 else -0.15), lbl,
                 ha='center', va='bottom', fontweight='bold', fontsize=9)
plt.tight_layout(); plt.show()

# ── Top 5 News ────────────────────────────────────────────────
print(f"\n{'='*70}")
print(f"  📰  TOP 5 NEWS — {SELECTED_NAME}")
print(f"{'='*70}")
if SENTIMENT_HEADLINES:
    for i, art in enumerate(SENTIMENT_HEADLINES[:5], 1):
        tone = "🟢" if art['sentiment'] > 0.05 else ("🔴" if art['sentiment'] < -0.05 else "🟡")
        print(f"  {i}. [{art['date']}] {tone}  Sentiment: {art['sentiment']:+.3f}  [{art['source']}]")
        print(f"     {art['headline'][:100]}")
        print()
else:
    print("  No headlines retrieved (NewsAPI/Finnhub keys not configured).")
print(f"{'='*70}")
print("\n  ⚠  Disclaimer: Research / educational tool only.")
print("     Do NOT use as sole basis for real financial decisions.")


## CELL 15 — Project Summary

In [ ]:
# ============================================================
# CELL 15: PROJECT SUMMARY
# ============================================================

print("\n" + "="*72)
print("  PROJECT SUMMARY — Nifty 50 Stock Recommendation via Deep RL")
print("="*72)
print(f"\n  Stock Analysed  : {SELECTED_NAME} ({SELECTED_TICKER})")
print(f"  RL Algorithm    : PPO (Proximal Policy Optimization)")
print(f"  Training Scope  : All 10 Nifty stocks (generalised model)")
print(f"  Market          : National Stock Exchange, India (NSE)\n")
print(f"  Date Splits:")
print(f"    Train : {TRAIN_START}  →  {TRAIN_END}  (all 10 stocks)")
print(f"    Val   : {VAL_START}  →  {VAL_END}  (all 10 stocks)")
print(f"    Test  : {TEST_START}  →  {TEST_END}  ({SELECTED_TICKER} only)")
print(f"\n  PPO Hyperparameters:")
print(f"    Total Timesteps : {TOTAL_TIMESTEPS:,}")
print(f"    Learning Rate   : {LEARNING_RATE}")
print(f"    N Steps         : {N_STEPS}")
print(f"    Batch Size      : {BATCH_SIZE}")
print(f"    N Epochs        : {N_EPOCHS}")
print(f"    Gamma           : {GAMMA}   GAE Lambda : {GAE_LAMBDA}")
print(f"    Clip Range      : {CLIP_RANGE}    Entropy Coef: {ENT_COEF}")
print(f"    VF Coef         : {VF_COEF}    Max Grad Norm: {MAX_GRAD_NORM}")
print(f"    Net Architecture: [256, 256, 128] (policy + value)")
print(f"\n  Test Period Performance:")
for key in ['Sharpe Ratio','Sortino Ratio','Calmar Ratio',
            'Max Drawdown','Win Rate','Cumulative Return','Buy Precision']:
    print(f"    {key:<28}: PPO={test_metrics.get(key,'N/A'):>10}   B&H={bnh_test_m.get(key,'N/A'):>10}")
print(f"\n  Final Recommendation : {rec['final']}")
print(f"  Technical Score      : {rec['tech_score']:.1f}/100")
print(f"  Sentiment Score      : {rec['sent_score']:+.4f}")
print(f"  Market Regime        : {REGIME_LABEL}  ({REGIME_SCORE:.1f}/100)")
print("="*72)
print("\n  References:")
print("   - Schulman et al. (2017): Proximal Policy Optimization Algorithms")
print("   - Liu et al. (2020): FinRL: Deep RL for Automated Stock Trading")
print("   - Stable-Baselines3: https://stable-baselines3.readthedocs.io")
print("   - yfinance: https://github.com/ranaroussi/yfinance")
print("   - Finnhub: https://finnhub.io")
print("   - NewsAPI: https://newsapi.org")
print("\n✅ Project complete!")
